<a href="https://colab.research.google.com/github/MaxineMwangi/flyrank/blob/main/w03_data_contract_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaxineMwangi/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**How to use this version:** every `TODO` is something only you can fill in after looking at the real warehouse columns. Run the setup cells first, read the schema, then fill the TODOs.

## 0. Setup (run first)

*Connect to the Hugging Face warehouse with DuckDB. Token comes from the Colab Secret `HF_TOKEN` — never paste it in a cell.*

In [ ]:
!pip -q install duckdb huggingface_hub scikit-learn pandas

import duckdb, pandas as pd
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")          # Colab Secret, not a pasted string
REPO = "FlyRank/internship-warehouse"

con = duckdb.connect()
con.sql(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")

files = HfApi().list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN)
for f in files:
    print(f)

*Read the file list above. Pick the table/file(s) your lane uses and put the path(s) in `TABLE_PATHS` below. Then run the schema cell.*

In [ ]:
# ---- TODO: fill these in after reading the file list above ----
MONTH        = "2026-03"      # mid-panel month. Do NOT use the _sample table / June 2026 for label logic.
TABLE_PATHS  = ["hf://datasets/FlyRank/internship-warehouse/TODO_PATH.parquet"]
DATE_COL     = "TODO_date_column"          # the column that holds the row's date
ENTITY_KEYS  = ["TODO_key_1", "TODO_key_2"]  # columns that together should make one row unique (the grain you claim)
AVAIL_COL    = "TODO_availability_column"  # a boolean-ish column that says the row is usable / available
LABEL_COL    = "TODO_label_or_proxy_column"
TASK         = "regression"   # "regression" or "classification"
# ----------------------------------------------------------------

paths_sql = ", ".join(f"'{p}'" for p in TABLE_PATHS)
con.sql(f"CREATE OR REPLACE VIEW raw AS SELECT * FROM read_parquet([{paths_sql}])")
con.sql("DESCRIBE raw").df()

*Now cut the mid-panel month. Everything after this uses `slice_m`, so no query accidentally touches the sealed test month.*

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW slice_m AS
SELECT * FROM raw
WHERE strftime(CAST({DATE_COL} AS DATE), '%Y-%m') = '{MONTH}'
""")
con.sql("SELECT * FROM slice_m LIMIT 5").df()

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Contract answers (plain words, 5 total — 1–3 here, 4–5 in section 2):**

1. **What one row means for my lane:** TODO — e.g. "one row = one ___ on one ___"
2. **Table(s) I use:** TODO — name the table/file(s) from `TABLE_PATHS`
3. **Time window:** TODO — e.g. "all days in 2026-03; June 2026 is kept as a sealed test month"

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Window used:", MONTH)
con.sql(f"SELECT MIN(CAST({DATE_COL} AS DATE)) AS first_day, MAX(CAST({DATE_COL} AS DATE)) AS last_day FROM slice_m").df()

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Contract answers (continued):**

4. **What I predict or rank (label or proxy):** TODO — name `LABEL_COL` and say in one line why it is the label (or why it is only a proxy)
5. **One thing I deliberately exclude, and why:** TODO

| Bucket | Fields | Why |
|---|---|---|
| Feature | TODO | |
| Label | TODO | |
| Context (ids, dates, not fed to model) | TODO | |
| Excluded | TODO | TODO |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("DESCRIBE slice_m").df()

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain: is one row really what I said?
*If `n_rows` equals `n_distinct_keys`, the keys in `ENTITY_KEYS` are a true grain. If not, the grain claim above is wrong — fix the claim, not the query.*

In [ ]:
keys = ", ".join(ENTITY_KEYS)
q1 = f"""
SELECT
  COUNT(*) AS n_rows,
  (SELECT COUNT(*) FROM (SELECT DISTINCT {keys} FROM slice_m)) AS n_distinct_keys
FROM slice_m
"""
con.sql(q1).df()

**What Query 1 shows:** TODO — one sentence in your own words (does the grain hold?)

### Query 2 — My slice's row count and date span

In [ ]:
q2 = f"""
SELECT
  COUNT(*)                            AS n_rows,
  MIN(CAST({DATE_COL} AS DATE))       AS first_day,
  MAX(CAST({DATE_COL} AS DATE))       AS last_day,
  COUNT(DISTINCT CAST({DATE_COL} AS DATE)) AS n_days
FROM slice_m
"""
con.sql(q2).df()

**What Query 2 shows:** TODO — rows, dates, any missing days?

### Query 3 — Availability (filter with `IS TRUE`)
*Use `IS TRUE`, not `= TRUE` or a bare `WHERE col`. NULL is not the same as FALSE, and `IS TRUE` makes that explicit.*

In [ ]:
q3 = f"""
SELECT
  COUNT(*)                                          AS total_rows,
  COUNT(*) FILTER (WHERE {AVAIL_COL} IS TRUE)       AS rows_available,
  COUNT(*) FILTER (WHERE {AVAIL_COL} IS NULL)       AS rows_null,
  ROUND(100.0 * COUNT(*) FILTER (WHERE {AVAIL_COL} IS TRUE) / COUNT(*), 2) AS pct_available
FROM slice_m
"""
con.sql(q3).df()

**What Query 3 shows:** TODO — how many rows survive, and what that means for your lane.

### Five features (max) — each knowable at the decision moment
*Pick up to five columns. For each one write a line: "knowable at the decision moment because…". If you can't finish that sentence, it is not a feature.*

In [ ]:
# ---- TODO: choose up to 5 feature columns from the schema above ----
FEATURES = ["TODO_feat_1", "TODO_feat_2", "TODO_feat_3", "TODO_feat_4", "TODO_feat_5"]
assert len(FEATURES) <= 5, "five features max"

available_when = {
    "TODO_feat_1": "knowable at the decision moment because ...",
    "TODO_feat_2": "knowable at the decision moment because ...",
    "TODO_feat_3": "knowable at the decision moment because ...",
    "TODO_feat_4": "knowable at the decision moment because ...",
    "TODO_feat_5": "knowable at the decision moment because ...",
}
# --------------------------------------------------------------------

cols = ", ".join(ENTITY_KEYS + [DATE_COL] + FEATURES + [LABEL_COL])
df = con.sql(f"""
SELECT {cols} FROM slice_m
WHERE {AVAIL_COL} IS TRUE AND {LABEL_COL} IS NOT NULL
""").df()

print(df.shape)
for f, why in available_when.items():
    print(f"- {f}: {why}")
df.head()

### The trap — add ONE label-derived column on purpose, then remove it
*Step 1: score honestly. Step 2: add a column built from the label and watch the score jump toward perfect. Step 3: delete it and keep the honest number.*

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, roc_auc_score

def quick_score(frame, feature_cols):
    # time-ordered split inside the month: earlier days train, later days test
    frame = frame.sort_values(DATE_COL)
    cut = int(len(frame) * 0.7)
    tr, te = frame.iloc[:cut], frame.iloc[cut:]
    X_tr = tr[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    X_te = te[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    if TASK == "regression":
        m = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=0, n_jobs=-1)
        m.fit(X_tr, tr[LABEL_COL]); return r2_score(te[LABEL_COL], m.predict(X_te))
    m = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=0, n_jobs=-1)
    m.fit(X_tr, tr[LABEL_COL]); return roc_auc_score(te[LABEL_COL], m.predict_proba(X_te)[:, 1])

metric = "R2" if TASK == "regression" else "AUC"

# 1) honest score
honest = quick_score(df, FEATURES)
print(f"Honest {metric}: {honest:.3f}")

# 2) deliberate leak: a column derived straight from the label
df["LEAK_label_copy"] = pd.to_numeric(df[LABEL_COL], errors="coerce")
leaky = quick_score(df, FEATURES + ["LEAK_label_copy"])
print(f"Leaky  {metric}: {leaky:.3f}   <- suspiciously good = leakage")

# 3) delete it, keep the honest number
df = df.drop(columns=["LEAK_label_copy"])
print(f"After removing the leak, honest {metric} stays: {honest:.3f}")

**What the trap showed:** TODO — honest score vs leaky score, and in one line why the leaky column would never be available at the decision moment.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation of my slice:** TODO — e.g. what the month can't show, or which rows are missing/early-only. Use careful words: observed, measured, directional.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Optional: back your limitation with a number, e.g. share of NULLs per feature
df[FEATURES].isna().mean().round(3)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.